In [30]:
import pandas as pd
import os
import polars as pl
import plotly.express as px
import datetime
import plotly.graph_objects as go

DATA_PATH = '/kaggle/input/datasets/nikolasking/favorita'

In [2]:
train_sample = pd.read_csv(os.path.join(DATA_PATH, 'train.csv'), nrows=100000, parse_dates=['date'])
train_sample.isna().sum()

id                  0
date                0
store_nbr           0
item_nbr            0
unit_sales          0
onpromotion    100000
dtype: int64

In [3]:
train_sample.dtypes

id                      int64
date           datetime64[ns]
store_nbr               int64
item_nbr                int64
unit_sales            float64
onpromotion           float64
dtype: object

In [4]:
train_sample.head()

,id,date,store_nbr,item_nbr,unit_sales,onpromotion
0,0,2013-01-01,25,103665,7.0,NaN
1,1,2013-01-01,25,105574,1.0,NaN
2,2,2013-01-01,25,105575,2.0,NaN
3,3,2013-01-01,25,108079,1.0,NaN
4,4,2013-01-01,25,108701,1.0,NaN


In [11]:
def meta(df):
    df = pd.read_csv(os.path.join(DATA_PATH, f'{df}.csv'))
    print("Shape:", df.shape)
    print("\nТипы данных:\n", df.dtypes)
    print("\nПервые 5 строк:\n", df.head())
    print("\nПропуски: ", df.isna().sum().sum())
    print("\nУникальные значения:\n")
    for col in df.columns:
        print(f"{col}:", df[col].nunique())
        print()

datasets = ['stores', 'items', 'oil', 'holidays_events']
for i in datasets:
    print(f'=== {i} ===')
    meta(i)

=== stores ===
Shape: (54, 5)

Типы данных:
 store_nbr     int64
city         object
state        object
type         object
cluster       int64
dtype: object

Первые 5 строк:
    store_nbr           city                           state type  cluster
0          1          Quito                       Pichincha    D       13
1          2          Quito                       Pichincha    D       13
2          3          Quito                       Pichincha    D        8
3          4          Quito                       Pichincha    D        9
4          5  Santo Domingo  Santo Domingo de los Tsachilas    D        4

Пропуски:  0

Уникальные значения:

store_nbr: 54

city: 22

state: 16

type: 5

cluster: 17

=== items ===
Shape: (4100, 4)

Типы данных:
 item_nbr       int64
family        object
class          int64
perishable     int64
dtype: object

Первые 5 строк:
    item_nbr        family  class  perishable
0     96995     GROCERY I   1093           0
1     99197     GROCERY I   1067

### >>>

Бегло пробежались. Теперь начинаем EDA.

In [5]:
# Для удобства создаем пустой словарь, где будем хранить все загруженные датафреймы 
dataframes = {} # Ключ - имя файла (без .csv), значение - загруженные данные

filenames = ['train.csv', 'test.csv', 'stores.csv', 'items.csv', 'transactions.csv', 'oil.csv', 'holidays_events.csv']
base_dir = '/kaggle/input/datasets/nikolasking/favorita/'

for filename in filenames:
    file_path = f"{base_dir}{filename}"
    df_name = filename.replace('.csv', '')
    print(f"\nLoading {filename}...")

    lazy_df = pl.scan_csv(file_path)# Используем ленивую загрузку (scan_csv вместо read_csv) для экономии RAM
    # Если в данных есть колонка 'date' - преобразуем ее из строки в тип Date
    if 'date' in lazy_df.columns:
        lazy_df = lazy_df.with_columns(pl.col('date').str.to_date())
    
    # Специальная обработка для train.csv и test.csv
    if df_name in ['train', 'test'] and 'onpromotion' in lazy_df.columns: # Проверяем, какой тип данных у колонки 'onpromotion' в исходном файле
        inferred_schema_for_onpromotion = pl.scan_csv(file_path).schema.get('onpromotion')  # .schema.get() позволяет узнать тип без загрузки всех данных
        
        # Если onpromotion хранится как строка ('true'/'false') - преобразуем в булевый тип
        # Это важно для модели, так как булевые значения занимают меньше памяти
        if inferred_schema_for_onpromotion == pl.String:
            lazy_df = lazy_df.with_columns(
                pl.col('onpromotion').str.to_lowercase().eq(pl.lit('true')).alias('onpromotion')
            )
    
    # Наконец, выполняем все запланированные операции и загружаем данные в память
    dataframes[df_name] = lazy_df.collect() # .collect() материализует ленивый датафрейм
    
    print(f"\Первые 5 строк {df_name}:")
    print(dataframes[df_name].head())
    # Выводим схему (типы данных) для проверки корректности загрузки
    print(f"\nSchema of {df_name}:")
    print(dataframes[df_name].schema)


Loading train.csv...


/tmp/ipykernel_55/2107118523.py:12: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  if 'date' in lazy_df.columns:
/tmp/ipykernel_55/2107118523.py:15: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  if df_name in ['train', 'test'] and 'onpromotion' in lazy_df.columns:
/tmp/ipykernel_55/2107118523.py:16: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  inferred_schema_for_onpromotion = pl.scan_csv(file_path).schema.get('onpromotion')


\Первые 5 строк train:
shape: (5, 6)
┌─────┬────────────┬───────────┬──────────┬────────────┬─────────────┐
│ id  ┆ date       ┆ store_nbr ┆ item_nbr ┆ unit_sales ┆ onpromotion │
│ --- ┆ ---        ┆ ---       ┆ ---      ┆ ---        ┆ ---         │
│ i64 ┆ date       ┆ i64       ┆ i64      ┆ f64        ┆ bool        │
╞═════╪════════════╪═══════════╪══════════╪════════════╪═════════════╡
│ 0   ┆ 2013-01-01 ┆ 25        ┆ 103665   ┆ 7.0        ┆ null        │
│ 1   ┆ 2013-01-01 ┆ 25        ┆ 105574   ┆ 1.0        ┆ null        │
│ 2   ┆ 2013-01-01 ┆ 25        ┆ 105575   ┆ 2.0        ┆ null        │
│ 3   ┆ 2013-01-01 ┆ 25        ┆ 108079   ┆ 1.0        ┆ null        │
│ 4   ┆ 2013-01-01 ┆ 25        ┆ 108701   ┆ 1.0        ┆ null        │
└─────┴────────────┴───────────┴──────────┴────────────┴─────────────┘

Schema of train:
Schema({'id': Int64, 'date': Date, 'store_nbr': Int64, 'item_nbr': Int64, 'unit_sales': Float64, 'onpromotion': Boolean})

Loading test.csv...
\Первые 5 строк test:

In [6]:
train_path = f"{base_dir}train.csv"
train_lazy = pl.scan_csv(train_path).with_columns(
    pl.col('date').str.to_date()
)

# Группируем данные по дате и суммируем продажи
daily_sales_lazy = train_lazy.group_by('date').agg(pl.col('unit_sales').sum().alias('total_sales')).sort('date')

# Материализуем результат (collect): выполняем все запланированные операции
# и загружаем агрегированные данные в память
daily_sales = daily_sales_lazy.collect()

print(daily_sales.head())
print("\nSchema of daily_sales:")
print(daily_sales.schema)

shape: (5, 2)
┌────────────┬─────────────┐
│ date       ┆ total_sales │
│ ---        ┆ ---         │
│ date       ┆ f64         │
╞════════════╪═════════════╡
│ 2013-01-01 ┆ 2511.619    │
│ 2013-01-02 ┆ 496092.418  │
│ 2013-01-03 ┆ 361429.231  │
│ 2013-01-04 ┆ 354459.677  │
│ 2013-01-05 ┆ 477350.121  │
└────────────┴─────────────┘

Schema of daily_sales:
Schema({'date': Date, 'total_sales': Float64})


In [10]:
# Добавляем скользящее среднее к данным о ежедневных продажах
# rolling_mean(window_size=7) считает среднее за последние 7 дней для каждой даты
# Для первых 6 дней результат будет null (недостаточно данных для окна)

daily_sales = daily_sales.with_columns(
    pl.col('total_sales').rolling_mean(window_size=7).alias('rolling_avg_sales')
)

# Определяем дату землетрясения в Эквадоре (важное событие, повлиявшее на продажи)
# Это историческая дата из описания датасета
earthquake_date_obj = datetime.date(2016, 4, 16)

# Преобразуем дату для Plotly
# Plotly ожидает строки для позиционирования элементов на графике
earthquake_date_str = earthquake_date_obj.isoformat()

# Create the line plot
fig = px.line(
    daily_sales,
    x='date',
    y=['total_sales', 'rolling_avg_sales'],
    title='Среднее за последние 7 дней ',
    labels={
        'date': 'Date',
        'value': 'Sales',
        'variable': 'Metric'
    }
)

fig.add_shape(
    type="line",
    x0=earthquake_date_str,
    x1=earthquake_date_str,
    y0=0,
    y1=1,
    xref="x",
    yref="paper",
    line=dict(
        dash="dash",
        color="red",
    )
)

fig.add_annotation(
    x=earthquake_date_str,
    y=1,
    xref="x",
    yref="paper",
    text=f'Earthquake ({earthquake_date_str})',
    showarrow=False,
    xanchor="right",
    yanchor="top",
    yshift=10,
    font=dict(
        color="red"
    )
)

fig.show()

print(daily_sales.head())

shape: (5, 3)
┌────────────┬─────────────┬───────────────────┐
│ date       ┆ total_sales ┆ rolling_avg_sales │
│ ---        ┆ ---         ┆ ---               │
│ date       ┆ f64         ┆ f64               │
╞════════════╪═════════════╪═══════════════════╡
│ 2013-01-01 ┆ 2511.619    ┆ null              │
│ 2013-01-02 ┆ 496092.418  ┆ null              │
│ 2013-01-03 ┆ 361429.231  ┆ null              │
│ 2013-01-04 ┆ 354459.677  ┆ null              │
│ 2013-01-05 ┆ 477350.121  ┆ null              │
└────────────┴─────────────┴───────────────────┘


Выводы из графика:

Заметен рост продаж в целом за период

После землетрясения 2016-04-1 - резкий рост.

Перовго января - продажи сильно падают.

До 2015 года основные пики приходились на весна-лето, после - основной месяц продаж - декабрь (судя по графику)

### Анализ сезонности на неделе

In [11]:


daily_sales = daily_sales.with_columns(
    pl.col('date').dt.weekday().alias('day_of_week')
)

# Aggregate average sales by day of the week
avg_sales_by_day = daily_sales.group_by('day_of_week').agg(
    pl.col('total_sales').mean().alias('average_sales')
).sort('day_of_week')

print("daily_sales with day_of_week:")
print(daily_sales.head())
print("\n average sales by day of week:")
print(avg_sales_by_day.head())
print("\nSchema of average sales by day of week:")
print(avg_sales_by_day.schema)

daily_sales with day_of_week:
shape: (5, 4)
┌────────────┬─────────────┬───────────────────┬─────────────┐
│ date       ┆ total_sales ┆ rolling_avg_sales ┆ day_of_week │
│ ---        ┆ ---         ┆ ---               ┆ ---         │
│ date       ┆ f64         ┆ f64               ┆ i8          │
╞════════════╪═════════════╪═══════════════════╪═════════════╡
│ 2013-01-01 ┆ 2511.619    ┆ null              ┆ 2           │
│ 2013-01-02 ┆ 496092.418  ┆ null              ┆ 3           │
│ 2013-01-03 ┆ 361429.231  ┆ null              ┆ 4           │
│ 2013-01-04 ┆ 354459.677  ┆ null              ┆ 5           │
│ 2013-01-05 ┆ 477350.121  ┆ null              ┆ 6           │
└────────────┴─────────────┴───────────────────┴─────────────┘

 average sales by day of week:
shape: (5, 2)
┌─────────────┬───────────────┐
│ day_of_week ┆ average_sales │
│ ---         ┆ ---           │
│ i8          ┆ f64           │
╞═════════════╪═══════════════╡
│ 1           ┆ 617479.929934 │
│ 2           ┆ 569916.48

In [13]:
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

day_lookup_df = pl.DataFrame({
    'day_of_week': [i + 1 for i, _ in enumerate(day_names)],
    'day_name': day_names
})

avg_sales_by_day = avg_sales_by_day.join(day_lookup_df, on='day_of_week', how='left')
# Визуализируем
fig = px.bar(
    avg_sales_by_day,
    x='day_name',
    y='average_sales',
    title='Средняя цена по дням недели:',
    labels={
        'day_name': 'Day of the Week',
        'average_sales': 'Average Total Sales'
    },
    color='average_sales',
    color_continuous_scale=px.colors.sequential.Viridis
)

fig.update_layout(xaxis={'categoryorder':'array', 'categoryarray':day_names})

fig.show()

print("Средняя цена по дням недели:")
print(avg_sales_by_day)

Средняя цена по дням недели:
shape: (7, 4)
┌─────────────┬───────────────┬───────────┬────────────────┐
│ day_of_week ┆ average_sales ┆ day_name  ┆ day_name_right │
│ ---         ┆ ---           ┆ ---       ┆ ---            │
│ i8          ┆ f64           ┆ str       ┆ str            │
╞═════════════╪═══════════════╪═══════════╪════════════════╡
│ 1           ┆ 617479.929934 ┆ Monday    ┆ Monday         │
│ 2           ┆ 569916.489993 ┆ Tuesday   ┆ Tuesday        │
│ 3           ┆ 593244.190965 ┆ Wednesday ┆ Wednesday      │
│ 4           ┆ 505256.688612 ┆ Thursday  ┆ Thursday       │
│ 5           ┆ 579573.986125 ┆ Friday    ┆ Friday         │
│ 6           ┆ 772147.340838 ┆ Saturday  ┆ Saturday       │
│ 7           ┆ 825218.075771 ┆ Sunday    ┆ Sunday         │
└─────────────┴───────────────┴───────────┴────────────────┘


Из графика видно, что самыми высокими средними продажами обладают воскресенье и суббота.
Четверг считается днем с самыми низкими средними продажами. В целом, в продажах прослеживается четкая и сильная сезонность за неделю, с отчетливым пиком в выходные дни (суббота и воскресенье) и заметным снижением в будние дни, особенно в четверг. Эта закономерность свидетельствует о том, что покупатели чаще совершают покупки и тратят больше денег в выходные, что является общей тенденцией розничной торговли.

### Ежемесячный анализ сезонности

In [15]:
daily_sales = daily_sales.with_columns(
    pl.col('date').dt.month().alias('month') # Extract month (1-12)
)

avg_sales_by_month = daily_sales.group_by('month').agg(
    pl.col('total_sales').mean().alias('average_sales')
).sort('month')

print("Проверим, что все добавилось:")
print(daily_sales.head())
print()
print(avg_sales_by_month.head(20)) # <- должно быть 12

print("\nSchema of average sales by month:")
print(avg_sales_by_month.schema)

Проверим, что все добавилось:
shape: (5, 5)
┌────────────┬─────────────┬───────────────────┬─────────────┬───────┐
│ date       ┆ total_sales ┆ rolling_avg_sales ┆ day_of_week ┆ month │
│ ---        ┆ ---         ┆ ---               ┆ ---         ┆ ---   │
│ date       ┆ f64         ┆ f64               ┆ i8          ┆ i8    │
╞════════════╪═════════════╪═══════════════════╪═════════════╪═══════╡
│ 2013-01-01 ┆ 2511.619    ┆ null              ┆ 2           ┆ 1     │
│ 2013-01-02 ┆ 496092.418  ┆ null              ┆ 3           ┆ 1     │
│ 2013-01-03 ┆ 361429.231  ┆ null              ┆ 4           ┆ 1     │
│ 2013-01-04 ┆ 354459.677  ┆ null              ┆ 5           ┆ 1     │
│ 2013-01-05 ┆ 477350.121  ┆ null              ┆ 6           ┆ 1     │
└────────────┴─────────────┴───────────────────┴─────────────┴───────┘

shape: (12, 2)
┌───────┬───────────────┐
│ month ┆ average_sales │
│ ---   ┆ ---           │
│ i8    ┆ f64           │
╞═══════╪═══════════════╡
│ 1     ┆ 609302.427826 │
│ 2

In [16]:
month_names = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']

month_lookup_df = pl.DataFrame({
    'month': [i + 1 for i, _ in enumerate(month_names)], # 1 for January, ..., 12 for December
    'month_name': month_names
})

avg_sales_by_month = avg_sales_by_month.join(month_lookup_df, on='month', how='left')

fig = px.bar(
    avg_sales_by_month,
    x='month_name',
    y='average_sales',
    title='Средний общий объем продаж по месяцам',
    labels={
        'month_name': 'Month',
        'average_sales': 'Average Total Sales'
    },
    color='average_sales',
    color_continuous_scale=px.colors.sequential.Viridis
)

fig.update_layout(xaxis={'categoryorder':'array', 'categoryarray':month_names})

fig.show()

print("средний общий объем продаж по месяцам:")
print(avg_sales_by_month)

средний общий объем продаж по месяцам:
shape: (12, 3)
┌───────┬───────────────┬────────────┐
│ month ┆ average_sales ┆ month_name │
│ ---   ┆ ---           ┆ ---        │
│ i8    ┆ f64           ┆ str        │
╞═══════╪═══════════════╪════════════╡
│ 1     ┆ 609302.427826 ┆ January    │
│ 2     ┆ 571895.137866 ┆ February   │
│ 3     ┆ 627280.426773 ┆ March      │
│ 4     ┆ 604434.377091 ┆ April      │
│ 5     ┆ 608923.18723  ┆ May        │
│ …     ┆ …             ┆ …          │
│ 8     ┆ 600520.495226 ┆ August     │
│ 9     ┆ 645613.966011 ┆ September  │
│ 10    ┆ 645808.922815 ┆ October    │
│ 11    ┆ 669464.117255 ┆ November   │
│ 12    ┆ 808541.583939 ┆ December   │
└───────┴───────────────┴────────────┘


На основе гистограммы, можно сделать вывод о том, что в данных о продажах прослеживается четкая сезонная закономерность.
Декабрь выделяется как месяц со значительно более высокими средними продажами, что свидетельствует о сильном влиянии праздничных покупок. В ноябре также наблюдаются высокие продажи, вероятно, из-за предпраздничных покупок и мероприятий "Черной пятницы". Сентябрь и октябрь также демонстрируют относительно более высокие продажи по сравнению с серединой года.

В феврале стабильно наблюдаются самые низкие средние продажи, за которыми следуют январь и август. Это может быть связано с замедлением продаж после праздников и, возможно, снижением рекламной активности.

### Анализ влияния праздников

In [17]:
train_df = dataframes['train']
holidays_df = dataframes['holidays_events']

holidays_for_join = holidays_df.select(['date', 'type', 'transferred'])

train_with_holidays = train_df.join(
    holidays_for_join,
    on='date',
    how='left'
)

train_with_holidays = train_with_holidays.with_columns(
    (pl.col('type').is_not_null() & (~pl.col('transferred').fill_null(False)))
    .alias('is_holiday')
)

dataframes['train_with_holidays'] = train_with_holidays

print(train_with_holidays.head())
print("\nSchema of train_with_holidays:")
print(train_with_holidays.schema)

shape: (5, 9)
┌─────┬────────────┬───────────┬──────────┬───┬─────────────┬─────────┬─────────────┬────────────┐
│ id  ┆ date       ┆ store_nbr ┆ item_nbr ┆ … ┆ onpromotion ┆ type    ┆ transferred ┆ is_holiday │
│ --- ┆ ---        ┆ ---       ┆ ---      ┆   ┆ ---         ┆ ---     ┆ ---         ┆ ---        │
│ i64 ┆ date       ┆ i64       ┆ i64      ┆   ┆ bool        ┆ str     ┆ bool        ┆ bool       │
╞═════╪════════════╪═══════════╪══════════╪═══╪═════════════╪═════════╪═════════════╪════════════╡
│ 0   ┆ 2013-01-01 ┆ 25        ┆ 103665   ┆ … ┆ null        ┆ Holiday ┆ false       ┆ true       │
│ 1   ┆ 2013-01-01 ┆ 25        ┆ 105574   ┆ … ┆ null        ┆ Holiday ┆ false       ┆ true       │
│ 2   ┆ 2013-01-01 ┆ 25        ┆ 105575   ┆ … ┆ null        ┆ Holiday ┆ false       ┆ true       │
│ 3   ┆ 2013-01-01 ┆ 25        ┆ 108079   ┆ … ┆ null        ┆ Holiday ┆ false       ┆ true       │
│ 4   ┆ 2013-01-01 ┆ 25        ┆ 108701   ┆ … ┆ null        ┆ Holiday ┆ false       ┆ true     

In [18]:
avg_sales_holiday_impact = dataframes['train_with_holidays'].group_by('is_holiday').agg(
    pl.col('unit_sales').mean().alias('average_unit_sales')
).sort('is_holiday')

print("Средние продажи в праздничные и нерабочие дни:")
print(avg_sales_holiday_impact)

Средние продажи в праздничные и нерабочие дни:
shape: (2, 2)
┌────────────┬────────────────────┐
│ is_holiday ┆ average_unit_sales │
│ ---        ┆ ---                │
│ bool       ┆ f64                │
╞════════════╪════════════════════╡
│ false      ┆ 8.458002           │
│ true       ┆ 9.128124           │
└────────────┴────────────────────┘


Результаты анализа влияния праздничных дней
Основаны на сравнении средних показателей продаж за единицу продукции:

В непраздничные дни средний показатель продаж за единицу продукции (is_holiday = false) составляет приблизительно 8,45 единиц.
Средний объем продаж в праздничные дни (is_holiday = true) составляет приблизительно 9,14 единиц.

Вывод: Праздничные дни, оказывают положительное влияние на продажи. В среднем продажи в праздничные дни выше (примерно 9,12 единиц) по сравнению с нерабочими днями (примерно 8,46 единиц). Это говорит о том, что клиенты, как правило, покупают больше товаров в праздничные дни, что может быть связано с увеличением свободного времени, специальными акциями или культурными традициями, связанными с этими днями.

### Анализ эффективности продвижения

In [19]:
train_df = dataframes['train']
test_df = dataframes['test']

null_count_train = train_df.select(pl.col('onpromotion').is_null().sum()).item()
total_rows_train = train_df.height
nan_percentage_train = (null_count_train / total_rows_train) * 100

null_count_test = test_df.select(pl.col('onpromotion').is_null().sum()).item()
total_rows_test = test_df.height
nan_percentage_test = (null_count_test / total_rows_test) * 100

print(f"Количество пропусков в train_df: {nan_percentage_train:.2f}%")
print(f"Количество пропусков test_df: {nan_percentage_test:.2f}%")

Количество пропусков в train_df: 17.26%
Количество пропусков test_df: 0.00%


In [20]:
train_with_holidays_filtered = dataframes['train_with_holidays'].filter(
    pl.col('onpromotion').is_not_null()
)

avg_sales_by_promotion = train_with_holidays_filtered.group_by('onpromotion').agg(
    pl.col('unit_sales').mean().alias('average_unit_sales')
).sort('onpromotion')

print("Средние продажи за единицу продукции с рекламными акциями и без них:")
print(avg_sales_by_promotion)

Средние продажи за единицу продукции с рекламными акциями и без них:
shape: (2, 2)
┌─────────────┬────────────────────┐
│ onpromotion ┆ average_unit_sales │
│ ---         ┆ ---                │
│ bool        ┆ f64                │
╞═════════════╪════════════════════╡
│ false       ┆ 8.107783           │
│ true        ┆ 13.440984          │
└─────────────┴────────────────────┘


Средний объем продаж за единицу продукции, на которую не распространяется промоакция, составляет приблизительно 8 единиц.
Средний объем продаж товаров, которые участвуют в акции, составляет приблизительно 13 единиц.

Вывод: видно, что рекламные акции очень эффективны для увеличения продаж. Средний объем продаж за единицу продукции значительно увеличивается (с ~8 до ~13 единиц), когда товары продаются по акции. Это важный инсайт.

### Отрицательный анализ продаж

In [21]:
train_df = dataframes['train']
# Фильтруем данные:
negative_sales_df = train_df.filter(pl.col('unit_sales') < 0)


negative_sales_count = negative_sales_df.height

# Получаем общее количество записей в тренировочном наборе
total_train_rows = train_df.height

percentage_negative_sales = (negative_sales_count / total_train_rows) * 100

# Print the calculated percentage
print(f"процент отрицательных продаж от общего числа записей в train_df: {percentage_negative_sales:.2f}%")

процент отрицательных продаж от общего числа записей в train_df: 0.01%


In [22]:
negative_sales_by_day = negative_sales_df.group_by('date').agg(pl.len().alias('negative_sales_count')).sort('date')

print("negative_sales_by_day:")
print(negative_sales_by_day.head())
print("\nSchema of negative_sales_by_day:")
print(negative_sales_by_day.schema)

negative_sales_by_day:
shape: (5, 2)
┌────────────┬──────────────────────┐
│ date       ┆ negative_sales_count │
│ ---        ┆ ---                  │
│ date       ┆ u32                  │
╞════════════╪══════════════════════╡
│ 2013-01-02 ┆ 1                    │
│ 2013-01-03 ┆ 5                    │
│ 2013-01-04 ┆ 4                    │
│ 2013-01-05 ┆ 1                    │
│ 2013-01-07 ┆ 1                    │
└────────────┴──────────────────────┘

Schema of negative_sales_by_day:
Schema({'date': Date, 'negative_sales_count': UInt32})


In [24]:
fig = px.line(
    negative_sales_by_day,
    x='date',
    y='negative_sales_count',
    title='Количество отрицательных продаж по дням',
    labels={
        'date': 'Date',
        'negative_sales_count': 'Number of Negative Sales Entries'
    }
)

fig.show()

print("negative_sales_by_day (after plotting):")
print(negative_sales_by_day.head())

negative_sales_by_day (after plotting):
shape: (5, 2)
┌────────────┬──────────────────────┐
│ date       ┆ negative_sales_count │
│ ---        ┆ ---                  │
│ date       ┆ u32                  │
╞════════════╪══════════════════════╡
│ 2013-01-02 ┆ 1                    │
│ 2013-01-03 ┆ 5                    │
│ 2013-01-04 ┆ 4                    │
│ 2013-01-05 ┆ 1                    │
│ 2013-01-07 ┆ 1                    │
└────────────┴──────────────────────┘


График, отражающий ежедневное количество отрицательных продаж за единицу продукции с течением времени, показывает, что эти случаи, как правило, являются незначительными. Не существует постоянных периодов с очень высокими показателями отрицательных продаж, а также сильных циклических закономерностей, таких как еженедельная или ежемесячная сезонность. Вместо этого отрицательные продажи проявляются в виде небольших изолированных всплесков на временной шкале. То есть скорее всего это ничего не значит, а просто ошибка. Учитывая, что отрицательные продажи составляют очень небольшой процент (0,01%) от общего числа записей (можно даже просто их удалить), этот визуальный анализ еще раз подтверждает, что это нечастые случаи, а не широко распространенная проблема.

### Анализ продаж скоропортящихся и не скоропортящихся продуктов

In [25]:
train_with_holidays_df = dataframes['train_with_holidays']
items_df = dataframes['items']

items_for_join = items_df.select(['item_nbr', 'perishable'])

train_with_perishability = train_with_holidays_df.join(
    items_for_join,
    on='item_nbr',
    how='left'
)
dataframes['train_with_perishability'] = train_with_perishability

print(train_with_perishability.head())
print("\nSchema of train_with_perishability:")
print(train_with_perishability.schema)

shape: (5, 10)
┌─────┬────────────┬───────────┬──────────┬───┬─────────┬─────────────┬────────────┬────────────┐
│ id  ┆ date       ┆ store_nbr ┆ item_nbr ┆ … ┆ type    ┆ transferred ┆ is_holiday ┆ perishable │
│ --- ┆ ---        ┆ ---       ┆ ---      ┆   ┆ ---     ┆ ---         ┆ ---        ┆ ---        │
│ i64 ┆ date       ┆ i64       ┆ i64      ┆   ┆ str     ┆ bool        ┆ bool       ┆ i64        │
╞═════╪════════════╪═══════════╪══════════╪═══╪═════════╪═════════════╪════════════╪════════════╡
│ 0   ┆ 2013-01-01 ┆ 25        ┆ 103665   ┆ … ┆ Holiday ┆ false       ┆ true       ┆ 1          │
│ 1   ┆ 2013-01-01 ┆ 25        ┆ 105574   ┆ … ┆ Holiday ┆ false       ┆ true       ┆ 0          │
│ 2   ┆ 2013-01-01 ┆ 25        ┆ 105575   ┆ … ┆ Holiday ┆ false       ┆ true       ┆ 0          │
│ 3   ┆ 2013-01-01 ┆ 25        ┆ 108079   ┆ … ┆ Holiday ┆ false       ┆ true       ┆ 0          │
│ 4   ┆ 2013-01-01 ┆ 25        ┆ 108701   ┆ … ┆ Holiday ┆ false       ┆ true       ┆ 1          │
└────

In [26]:
train_with_perishability = dataframes['train_with_perishability']

# Создаем новую колонку со взвешенными продажами
# Это важно для метрики NWRMSLE, где perishable товары имеют вес 1.25
train_with_perishability = train_with_perishability.with_columns(
    pl.when(pl.col('perishable') == 1)
    .then(pl.col('unit_sales') * 1.25)
    .otherwise(pl.col('unit_sales') * 1)
    .alias('weighted_unit_sales')
)

# Группируем по признаку скоропортящихся товаров (0 или 1)
avg_weighted_sales_by_perishability = train_with_perishability.group_by('perishable').agg(
    pl.col('weighted_unit_sales').mean().alias('average_weighted_unit_sales')
).sort('perishable')

print("train_with_perishability with weighted_unit_sales:")
print(train_with_perishability.head())
print("\nAverage weighted unit sales for perishable vs. non-perishable items:")
print(avg_weighted_sales_by_perishability)

train_with_perishability with weighted_unit_sales:
shape: (5, 11)
┌─────┬────────────┬───────────┬──────────┬───┬─────────────┬────────────┬────────────┬────────────┐
│ id  ┆ date       ┆ store_nbr ┆ item_nbr ┆ … ┆ transferred ┆ is_holiday ┆ perishable ┆ weighted_u │
│ --- ┆ ---        ┆ ---       ┆ ---      ┆   ┆ ---         ┆ ---        ┆ ---        ┆ nit_sales  │
│ i64 ┆ date       ┆ i64       ┆ i64      ┆   ┆ bool        ┆ bool       ┆ i64        ┆ ---        │
│     ┆            ┆           ┆          ┆   ┆             ┆            ┆            ┆ f64        │
╞═════╪════════════╪═══════════╪══════════╪═══╪═════════════╪════════════╪════════════╪════════════╡
│ 0   ┆ 2013-01-01 ┆ 25        ┆ 103665   ┆ … ┆ false       ┆ true       ┆ 1          ┆ 8.75       │
│ 1   ┆ 2013-01-01 ┆ 25        ┆ 105574   ┆ … ┆ false       ┆ true       ┆ 0          ┆ 1.0        │
│ 2   ┆ 2013-01-01 ┆ 25        ┆ 105575   ┆ … ┆ false       ┆ true       ┆ 0          ┆ 2.0        │
│ 3   ┆ 2013-01-01 ┆ 25  



Исходя из анализа видно, что существует значительная разница в средневзвешенных продажах за единицу продукции между скоропортящимися и непортящимися продуктами. Средневзвешенные продажи за единицу продукции для не скоропортящихся продуктов составляют приблизительно 7,80 единиц.
Средневзвешенный объем продаж скоропортящихся продуктов составляет приблизительно 13,5 единиц.Средневзвешенные продажи за единицу продукции у скоропортящихся продуктов значительно выше (почти в два раза) по сравнению с непортящимися продуктами. Это указывает на то, что их структура продаж, потребительский спрос и потенциально внешние факторы (такие как рекламные акции, сезонность или риск порчи) ведут себя по-разному.

### Землетрясение

In [27]:
earthquake_date = datetime.date(2016, 4, 16)

# 1 и 2 месяца после
start_date_2016 = earthquake_date - datetime.timedelta(days=30)
end_date_2016 = earthquake_date + datetime.timedelta(days=60)

# 2015 - аналогичный период
start_date_2015 = start_date_2016.replace(year=2015)
end_date_2015 = end_date_2016.replace(year=2015)

print(f"2016 Period: {start_date_2016} to {end_date_2016}")
print(f"2015 Analogous Period: {start_date_2015} to {end_date_2015}")

sales_2016_earthquake_period = daily_sales.filter(
    (pl.col('date') >= start_date_2016) & (pl.col('date') <= end_date_2016)
)

sales_2015_analogous_period = daily_sales.filter(
    (pl.col('date') >= start_date_2015) & (pl.col('date') <= end_date_2015)
)

print("\nHead of sales_2016_earthquake_period:")
print(sales_2016_earthquake_period.head())
print("\nHead of sales_2015_analogous_period:")
print(sales_2015_analogous_period.head())

2016 Period: 2016-03-17 to 2016-06-15
2015 Analogous Period: 2015-03-17 to 2015-06-15

Head of sales_2016_earthquake_period:
shape: (5, 5)
┌────────────┬─────────────┬───────────────────┬─────────────┬───────┐
│ date       ┆ total_sales ┆ rolling_avg_sales ┆ day_of_week ┆ month │
│ ---        ┆ ---         ┆ ---               ┆ ---         ┆ ---   │
│ date       ┆ f64         ┆ f64               ┆ i8          ┆ i8    │
╞════════════╪═════════════╪═══════════════════╪═════════════╪═══════╡
│ 2016-03-17 ┆ 580452.532  ┆ 739211.851286     ┆ 4           ┆ 3     │
│ 2016-03-18 ┆ 666756.819  ┆ 739478.037857     ┆ 5           ┆ 3     │
│ 2016-03-19 ┆ 895038.512  ┆ 739040.523857     ┆ 6           ┆ 3     │
│ 2016-03-20 ┆ 929605.582  ┆ 735415.559857     ┆ 7           ┆ 3     │
│ 2016-03-21 ┆ 690469.801  ┆ 738616.046714     ┆ 1           ┆ 3     │
└────────────┴─────────────┴───────────────────┴─────────────┴───────┘

Head of sales_2015_analogous_period:
shape: (5, 5)
┌────────────┬─────────────┬

In [28]:
sales_2016_earthquake_period = sales_2016_earthquake_period.with_columns(
    (pl.col('date').cast(pl.Int32) - pl.lit(start_date_2016, dtype=pl.Date).cast(pl.Int32)).alias('relative_date')
)
sales_2015_analogous_period = sales_2015_analogous_period.with_columns(
    (pl.col('date').cast(pl.Int32) - pl.lit(start_date_2015, dtype=pl.Date).cast(pl.Int32)).alias('relative_date')
)

comparison_df = sales_2016_earthquake_period.join(
    sales_2015_analogous_period,
    on='relative_date',
    how='outer',
    suffix='_2015'
).rename({
    'total_sales': 'total_sales_2016',
    'date': 'date_2016'
}).rename({
    'date_2015': 'date_2015_analogous'
})

comparison_df = comparison_df.with_columns(
    (pl.col('total_sales_2016') - pl.col('total_sales_2015')).alias('sales_difference')
)

print("Head of comparison_df:")
print(comparison_df.head())
print("\nSchema of comparison_df:")
print(comparison_df.schema)

Head of comparison_df:
shape: (5, 13)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ date_2016 ┆ total_sal ┆ rolling_a ┆ day_of_we ┆ … ┆ day_of_we ┆ month_201 ┆ relative_ ┆ sales_di │
│ ---       ┆ es_2016   ┆ vg_sales  ┆ ek        ┆   ┆ ek_2015   ┆ 5         ┆ date_2015 ┆ fference │
│ date      ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆ f64       ┆ f64       ┆ i8        ┆   ┆ i8        ┆ i8        ┆ i32       ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2016-03-1 ┆ 580452.53 ┆ 739211.85 ┆ 4         ┆ … ┆ 2         ┆ 3         ┆ 0         ┆ 165093.0 │
│ 7         ┆ 2         ┆ 1286      ┆           ┆   ┆           ┆           ┆           ┆ 16       │
│ 2016-03-1 ┆ 666756.81 ┆ 739478.03 ┆ 5         ┆ … ┆ 3         ┆ 3         ┆ 1         ┆ 268126.8 │
│ 8         ┆ 9         ┆ 7857      ┆           ┆   ┆

/tmp/ipykernel_55/3130205548.py:8: DeprecationWarning:

use of `how='outer'` should be replaced with `how='full'`.
(Deprecated in version 0.20.29)



In [31]:
# Вычисляем относительную дату землетрясения (количество дней от начала периода)
earthquake_relative_date = (earthquake_date - start_date_2016).days

# Explicitly convert Polars Series to NumPy arrays for plotting
x_data = comparison_df['relative_date'].to_numpy() # to_numpy() конвертирует Polars Series в NumPy array
sales_2016_data = comparison_df['total_sales_2016'].to_numpy()
sales_2015_data = comparison_df['total_sales_2015'].to_numpy()
sales_difference_data = comparison_df['sales_difference'].to_numpy()


# Create the line plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x_data,
    y=sales_2016_data,
    mode='lines',
    name='Total Sales 2016'
))

fig.add_trace(go.Scatter(
    x=x_data,
    y=sales_2015_data,
    mode='lines',
    name='Total Sales 2015'
))

fig.add_trace(go.Scatter(
    x=x_data,
    y=sales_difference_data,
    mode='lines',
    name='Sales Difference (2016 - 2015)',
    line=dict(dash='dot', color='purple')
))

# Add a vertical line for the earthquake date
fig.add_vline(
    x=earthquake_relative_date,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Earthquake ({earthquake_date})",
    annotation_position="top right",
    annotation_font_color="red"
)

fig.update_layout(
    title='Сравнение: 2016 (с Землетрясением) vs. 2015 (обычный период)',
    xaxis_title='Relative Day (from ' + str(start_date_2016) + ')',
    yaxis_title='Total Sales / Sales Difference',
    hovermode='x unified'
)

fig.show()


График, сравнивающий продажи в период землетрясения 2016 года с аналогичным периодом 2015 года, наглядно иллюстрирует разрушительные последствия землетрясения 16 апреля 2016 года.

Анализ показал, что после землетрясения 16 апреля 2016 года продажи резко выросли, что возможно связано с экстренным спросом на товары первой необходимости и гуманитарными закупками; судя по графику, пик пришелся на первые дни после катастрофы. График долго не мог стабилизироваться, но в конце года продажи постепенно вернулись к нормальным уровням, что подтверждается положительной разницей с аналогичным периодом 2015 года в первые недели после события.

In [30]:
# Быстрая проверка локальных праздников
print(dataframes['holidays_events']['locale'].value_counts())

# Проверка корреляции нефти с лагами (на примере daily_sales)
oil_test = daily_sales.join(dataframes['oil'], on='date', how='left').with_columns(
    pl.col('dcoilwtico').forward_fill()
)

for i in [1, 7, 14, 30]:
    corr = oil_test.select(
        pl.corr('total_sales', pl.col('dcoilwtico').shift(i))
    ).item()
    print(f"Correlation with oil lag {i}: {corr:.4f}")

shape: (3, 2)
┌──────────┬───────┐
│ locale   ┆ count │
│ ---      ┆ ---   │
│ str      ┆ u32   │
╞══════════╪═══════╡
│ National ┆ 174   │
│ Regional ┆ 24    │
│ Local    ┆ 152   │
└──────────┴───────┘
Correlation with oil lag 1: -0.6258
Correlation with oil lag 7: -0.6214
Correlation with oil lag 14: -0.6146
Correlation with oil lag 30: -0.6020


Корреляция -0.62 - это достаточно много. Минус говорит о том, что при падении цен на нефть продажи в магазинах исторически росли (или наоборот).

Лаги: Разница между лагом 1 и лагом 14 минимальна (всего 0.01), что говорит о том, что нефть - это скорее индикатор общего состояния экономики на длинной дистанции, а не ежедневный триггер.

### Summary:

* Общая динамика продаж: с 2013 по 2017 год наблюдается устойчивый тренд роста ежедневных продаж, что говорит о развитии бизнеса. В данных четко прослеживается недельная сезонность: продажи достигают пика в выходные дни (суббота и воскресенье) и снижаются в середине недели. Также присутствует годовая сезонность - продажи заметно растут в конце года (ноябрь-декабрь) и падают в начале года (январь-февраль).

* Влияние землетрясения:землетрясение 16 апреля 2016 года привело к резкому росту продаж. Вероятнее всего, это объясняется экстренным спросом на товары первой необходимости и гуманитарными закупками. Пик пришелся на первые дни после катастрофы. Это явная аномалия, и при моделировании ее нужно либо исключить из обучения (чтобы не запутать модель), либо добавить специальный флаг.

* Эффективность промо-акций: одним из самых сильных факторов влияния на продажи оказались рекламные акции. Товары, участвующие в промо, продаются в среднем на 65% лучше (13. единиц против 8), что делает этот признак критически важным для модели.

* Отрицательные продажи: составляют всего 0.01% от всех записей, распределены равномерно во времени и не имеют выраженной сезонности. Это позволяет безопасно заменить их на 0 при подготовке данных, упростив задачу модели.

* Скоропортящиеся vs обычные товары: средние продажи скоропортящихся товаров почти вдвое выше (13.56 против 7.80), что указывает на необходимость моделировать эти категории отдельно или как минимум учитывать их разные паттерны потребления.